In [ ]:
import pandas as pd
import requests
from datetime import datetime , timedelta
import os
from mecoda_minka import get_obs, get_dfs
import folium
from folium.plugins import HeatMap
from html2image import Html2Image

# 1. Construir un dataframe de métricas principales del proyecto y variación en el último mes.

El dataframe resultante tendrá esta forma:

```python
metric, number_today, number_in_last_month
observations, 1521, 23
observers, 124, 2
identifiers, 26, 0
species, 462, 5
```

* number_today = dato actual de observaciones, observadores, identificadores, especies.
* number_in_last_month = dato de fecha actual menos dato registrado 30 días antes (variación en los últimos 30 días).

Usamos llamadas a la API, porque son datos totales. 

Creamos un directorio "data" donde guardaremos todos los csv que vamos a generar. Este dataframe lo guardamos como "data/main_metrics.cvs", sin incluir los índices (index=False).


In [ ]:
def get_main_metrics(id_project):

    """
    Obtiene las métricas principales de un proyecto.
    :param id_project: ID del proyecto
    :return: DataFrame con las métricas principales
    """
    
    # Primero he definido las fechas que se utilizaran para obtener las metricas : Defino el mes actual y el mes anterior

    # MES ACTUAL
    today = datetime.today()
    this_month = today.month
   
    # MES ANTERIOR
    last_month_date = today - timedelta(days=30)
    last_month = last_month_date.month
    

    base_url = 'https://api.minka-sdg.org/v1'

  
    # METRICAS PARA EL MES ANTERIOR
  
    url_observations_last = f'{base_url}/observations?project_id={id_project}&month={last_month}&quality_grade=research'
    observations_results_last = requests.get(url_observations_last).json()['total_results']

    url_observers_last = f'{base_url}/observations/observers?project_id={id_project}&month={last_month}&quality_grade=research'
    observers_results_last = requests.get(url_observers_last).json()['total_results']

    url_identifiers_last = f'{base_url}/observations/identifiers?project_id={id_project}&month={last_month}&quality_grade=research'
    identifiers_results_last = requests.get(url_identifiers_last).json()['total_results']

    url_species_last = f'{base_url}/observations/species_counts?project_id={id_project}&month={last_month}&quality_grade=research'
    species_results_last = requests.get(url_species_last).json()['total_results']

    
    # METRICAS PARA EL MES ACTUAL 

    url_observations_today = f'{base_url}/observations?project_id={id_project}&month={this_month}&quality_grade=research'
    observations_results_today = requests.get(url_observations_today).json()['total_results']

    url_observers_today = f'{base_url}/observations/observers?project_id={id_project}&month={this_month}&quality_grade=research'
    observers_results_today = requests.get(url_observers_today).json()['total_results']

    url_identifiers_today = f'{base_url}/observations/identifiers?project_id={id_project}&month={this_month}&quality_grade=research'
    identifiers_results_today = requests.get(url_identifiers_today).json()['total_results']

    url_species_today = f'{base_url}/observations/species_counts?project_id={id_project}&month={this_month}&quality_grade=research'
    species_results_today = requests.get(url_species_today).json()['total_results']


    # Aqui genero el dataframe para almacenar los datos (TODOS LOS DATOS SON CON EL RESEARCH GRADE)

    df_main_metrics = pd.DataFrame({
        'Metric': ['Observations', 'Observers', 'Identifiers', 'Species'],
        'Last Month': [observations_results_last, observers_results_last, identifiers_results_last, species_results_last],
        'Current Month': [observations_results_today, observers_results_today, identifiers_results_today, species_results_today]
    })

    os.makedirs('data', exist_ok=True)
    df_main_metrics.to_csv('data/main_metrics.csv', index=False)


    return df_main_metrics

In [ ]:
df_main_metrics = get_main_metrics(264)

# 2. Evolución de las métricas principales

Construir un dataframe con esta forma:

```python
month, observations, observers, identifiers, species
2024-01, 185, 15, 7, 62
2024-02, 128, 3, 1, 32
...
```

Los datos no son acumulativos, son del mes en concreto. Lo sacaremos usando llamadas a la API.

Para ello puedes utilizar estas funciones, que te ayudarán a construirlo:

In [ ]:
import requests

API_PATH = "https://api.minka-sdg.org/v1"

def get_totals(project_id, year, month, kind="project", session=None):

    if session is None:
        session = requests.Session()

    # Define the API endpoints for the different metrics
    # and construct the URLs with the provided project ID and date range
    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"  

    # Make GET requests to the API endpoints and extract the total results
    total_obs = session.get(url_obs).json()["total_results"]
    total_part = session.get(url_part).json()["total_results"]
    total_ident = session.get(url_ident).json()["total_results"]
    total_spe = session.get(url_spe).json()["total_results"]

    return total_obs, total_part, total_ident, total_spe

In [ ]:
from datetime import datetime
import calendar

def get_month_list(years: list) -> dict:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
            
    return meses

In [ ]:
meses = get_month_list(range(2022, datetime.now().year + 1))
meses

In [ ]:
# Para cada elemento de la lista, podemos sacar el año y el mes
meses[0].split("-")[0]  # Año
meses[0].split("-")[1]  # Mes

In [ ]:
# Ejemplo de uso con el primer mes

get_totals(
    project_id=264,
    year=int(meses[0].split("-")[0]),
    month=int(meses[0].split("-")[1]),
    kind="project"
)

Esto nos devuelve un diccionario con los meses como clave y el último día del mes como valor. Así podemos usarlo con la función anterior:

Ahora hay que unir las dos funciones para sacar cada mes y de cada mes sacar los valores de las métricas. Eso nos da los resultados de un mes, que podemos guardar en un diccionario. Y luego unimos los diccionarios de cada mes en una lista de todos los meses. Te pongo debajo un ejemplo de uso con un mes.

In [ ]:
# Ejemplo de proceso con un mes

# Creamos una lista vacía para almacenar los resultados de cada mes
total_metrics = []

# Te enseño cómo construirlo con el primer mes de la lista meses
mes = meses[0]
year = mes.split("-")[0]
month = mes.split("-")[1]
total_obs, total_spe, total_part, total_ident = get_totals(
    project_id=264, year=year, month=month, kind="project"
)

# Creamos un diccionario con los resultados del mes
# y lo añadimos a la lista de métricas totales
total_for_month = {}
total_for_month["month"] = meses[0]
total_for_month["total_obs"] = total_obs
total_for_month["total_spe"] = total_spe
total_for_month["total_part"] = total_part
total_for_month["total_ident"] = total_ident
total_metrics.append(total_for_month)

# Creamos un DataFrame a partir de la lista de métricas totales
df_monthly = pd.DataFrame(total_metrics)

Ahora hay que crear una función que itere por todos los elementos de la lista meses desde el inicio de MINKA y saque los datos para cada mes usando get_totals a un diccionario, los acumule en la lista y la lista la convierta a un dataframe.

Es decir, para cada elemento de los meses, usamos get_totals para sacar las métricas y las almacenamos. Si lo ves complicado lo hacemos juntos.

Ese dataframe lo guardamos como "data/monthly_metrics.csv".

In [ ]:
# Aqui he copiado y pegado tal cual el script de ejemplo con la finalidad de obtener las requests a la API

API_PATH = "https://api.minka-sdg.org/v1"

def get_totals(project_id, year, month, kind="project", session=None):
    if session is None:
        session = requests.Session()

    if kind == "project":
        url_obs = f"{API_PATH}/observations?project_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?project_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?project_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?project_id={project_id}&month={month}&year={year}"
    elif kind == "place":
        url_obs = f"{API_PATH}/observations?place_id={project_id}&month={month}&year={year}"
        url_spe = f"{API_PATH}/observations/species_counts?place_id={project_id}&month={month}&year={year}"
        url_part = f"{API_PATH}/observations/observers?place_id={project_id}&month={month}&year={year}"
        url_ident = f"{API_PATH}/observations/identifiers?place_id={project_id}&month={month}&year={year}"

    total_obs = session.get(url_obs).json().get("total_results", 0)
    total_part = session.get(url_part).json().get("total_results", 0)
    total_ident = session.get(url_ident).json().get("total_results", 0)
    total_spe = session.get(url_spe).json().get("total_results", 0)

    return total_obs, total_part, total_ident, total_spe

In [ ]:
# Aqui se crea la lista de meses tal y como se propone en  el ejemplo

def get_month_list(years: list) -> list:
    current_year = datetime.now().year
    current_month = datetime.now().month
    meses = []

    for year in years:
        max_month = 12 if year < current_year else current_month
        for month in range(1, max_month + 1):
            meses.append(f"{year}-{str(month).zfill(2)}")
    return meses

In [ ]:
# En esta casilla se genera la funcion que recorre los meses y obtiene las metricas de cada uno de ellos

def build_monthly_metrics(project_id):
    meses = get_month_list(range(2022, datetime.now().year + 1))
    total_metrics = []

    session = requests.Session()

    for mes in meses:
        year = int(mes.split("-")[0])
        month = int(mes.split("-")[1])

        # Agrego un print para saber si la evolución de los meses se esta ejecutando de forma correcta 

        print(f"Procesando {mes}...")

        # Aqui me ayudo ChatGPT, diciendo que estas lineas eran utiles para que si hay algun error no rompa todo el codigo y se almacene en la variable e

        try:
            total_obs, total_part, total_ident, total_spe = get_totals(
                project_id=project_id, year=year, month=month, kind="project", session=session
            )
        except Exception as e:
            print(f"Error procesando {mes}: {e}")
            continue


        # Generamos el diccionario para posteriormente añadirlos a la lista total_metrics 

        total_for_month = {
            "month": mes,
            "observations": total_obs,
            "observers": total_part,
            "identifiers": total_ident,
            "species": total_spe
        }

        total_metrics.append(total_for_month)

    df_monthly = pd.DataFrame(total_metrics)


    os.makedirs("data", exist_ok=True)
    df_monthly.to_csv("data/monthly_metrics.csv", index=False)

    return df_monthly

In [ ]:
df_monthly = build_monthly_metrics(264)

In [ ]:
observations = get_obs(id_project=264, grade="research")
df_obs, df_photos = get_dfs(observations)

os.makedirs("data", exist_ok=True)
df_obs.to_csv("data/observations.csv", index=False)
df_photos.to_csv("data/photos.csv", index=False)

# 3. Taxonomías

Descargamos todas las observaciones del proyecto, usando mecoda_minka. Generamos los dataframes de observaciones y de fotos. A partir de df_obs creamos una función que nos permita ver el número de observaciones por reino, filo, clase... Este rango se tiene que poder indicar como parámetro, para usar la misma función para cualquier rango.

Guardamos df_obs y df_photos como csv en la carpeta `data`. Y no guardamos los índices, como en los casos anteriores.

Creamos la función. Ten en cuenta las columnas de los rangos que tiene el dataframe de df_obs. En las columnas "kingdom", "phylum", "class",... están los rangos taxonómicos superiores a la identificación. El "taxon_name" es el identificado en la observación, y el "taxon_rank" el rango de la identificación. En las otras columnas están los rangos superiores. Esto lo hacemos juntos, que es un poco complicado de explicar por escrito.

In [ ]:
def get_taxon_count(df_obs, rank_level):
    """
    Obtiene el conteo de taxones por nivel de rango.
    :param df_obs: DataFrame con las observaciones
    :param rank_level: Nivel de rango (kingdom, phylum, class, order, family, genus, species)
    :return: DataFrame con el conteo de taxones
    """

    if rank_level not in df_obs.columns:
        raise ValueError(f"'{rank_level}' esta columna no existe en el DataFrame.")

    df_taxon_counts = df_obs[rank_level].value_counts().reset_index()
    df_taxon_counts.columns = [rank_level, "observation_count"]

    return df_taxon_counts

In [ ]:
get_taxon_count(df_obs, "kingdom")

In [ ]:
get_taxon_count(df_obs, "phylum")

In [ ]:
get_taxon_count(df_obs, "class")

# 4. Especies vistas por primera vez en el proyecto desde el último informe (últimos 30 días)

A partir del df_obs podemos sacar este dato fácilmente. Toma el dataframe, ordénalo por fecha de observación, en orden ascendente (las primeras observaciones estarán más arriba). Ahora quédate solo con las primeras observaciones de cada especie. Es decir:
* Seleccionamos aquellas observaciones que hayan llegado al nivel de especie (columna "taxon_rank" == "species").
* Nos quedamos con la primera observación de cada especie, usando drop_duplicates()
```python
df_first = df_obs.drop_duplicates(subset=["taxon_name"], keep="first")
```
* Así nos quedaremos con la primera observación de cada especie. Ahora filtramos de esta tabla las que tengan fecha de observación mayor a hoy menos 30 días (vistas en los últimos 30 días).

Esas serán las especies nuevas observadas en los últimos 30 días.

Primero haz el proceso y luego lo conviertes a una función. Es decir, carga el df_obs y haz los pasos con él, cuando te haya salido ya lo conviertes en función.

In [ ]:
def get_new_species(df_obs, last_days=30):
    """
    Obtiene las nuevas especies observadas en los últimos días.
    :param df_obs: DataFrame con las observaciones
    :param last_days: Número de días para considerar una especie como nueva
    :return: DataFrame con las nuevas especies
    """
    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")

    df_species = df_obs[df_obs["taxon_rank"] == "species"]

    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    
    df_new_species = df_first[df_first["observed_on"] > cutoff_date]

    return df_new_species

In [ ]:
new_spe = get_new_species(df_obs)
new_spe[["taxon_name", "observed_on", "user_login"]].head()

Función para sacar una foto de las nuevas especies

In [ ]:
def get_photos_new_species(df_new_species, df_photos):
    # El dataframe df_new_species tiene las especies nuevas, con el id de cada observación
    # Filtramos el dataframe de fotos para quedarnos solo con las fotos de las especies nuevas, las de los ids de esas observaciones.
    # Puedes utilizar el método isin() de pandas para filtrar el dataframe df_photos
    # df_photos['id'].isin(df_new_species['id'])
    # Investiga el método isin() y cómo se utiliza para filtrar un dataframe
    # Nos quedaríamos solo con una foto para cada especie nueva, así que podemos usar el método drop_duplicates() de pandas con subset(['id'])
    
    # Filtrar las fotos cuyas observaciones están en df_new_species
    df_photos_new_species = df_photos[df_photos['id'].isin(df_new_species['id'])]

    # Eliminar duplicados para quedarnos con una foto por observación
    df_photos_new_species = df_photos_new_species.drop_duplicates(subset=['id'])

    return df_photos_new_species

In [ ]:
df_new_species = get_new_species(df_obs, last_days=30)
df_new_species

In [ ]:
from datetime import datetime, timedelta
import pandas as pd

def get_new_species_photo_snapshot(df_obs, df_photos, last_days=30):

    df_obs["observed_on"] = pd.to_datetime(df_obs["observed_on"], errors="coerce")
    df_species = df_obs[df_obs["taxon_rank"] == "species"]
    df_species_sorted = df_species.sort_values(by="observed_on")
    df_first = df_species_sorted.drop_duplicates(subset=["taxon_name"], keep="first")

    cutoff_date = datetime.today() - timedelta(days=last_days)
    df_new_species = df_first[df_first["observed_on"] > cutoff_date]

    # Esta parte la he hecho con ChatGPT porque me daba un  error que no estaba entendiendo y a añadido la funcion map
    # lo que hace es agregar el url a la foto correspondiente basandose en el id
    df_photos_unique = df_photos.drop_duplicates(subset="id")
    photo_map = df_photos_unique.set_index("id")["photos_medium_url"]
    df_new_species["photos_medium_url"] = df_new_species["id"].map(photo_map)


    os.makedirs("data", exist_ok=True)
    df_new_species[["taxon_name", "observed_on", "user_login", "photos_medium_url"]].to_csv("data/new_species_photo.csv", index=False)
    
    return df_new_species[["taxon_name", "observed_on", "user_login", "photos_medium_url"]].reset_index(drop=True)


In [ ]:
get_new_species_photo_snapshot(df_obs,df_photos, last_days=30)

Con esto estaríamos creando las funciones para extraer los datos. Luego estarían las de crear los gráficos y montar el informe.

# 5. Mapa calor para densidad de observaciones

Crear la función que toma un dataframe con el formato de df_obs (con esos nombres de columna, "latitude", "longitude") y lo mapee en el mapa de calor. Ese dataframe puede estar con las observaciones totales, filtrato por un kingdom, por un usuario, por un mes, o por lo que sea, pero no le afecta a la función, que lo hará siempre igual sobre un dataframe con las mismas columnas.

In [ ]:
def get_heat_map(df_obs, zoom_start:int, output_name:str="heatmap.png"):

    df_valid = df_obs.dropna(subset=["latitude", "longitude"])
    heat_data = df_valid[["latitude", "longitude"]].values.tolist()

    mean_lat = df_valid["latitude"].mean()
    mean_lon = df_valid["longitude"].mean()
   

    m = folium.Map(location=[mean_lat, mean_lon], zoom_start=zoom_start)
    HeatMap(heat_data).add_to(m)

    os.makedirs("figures", exist_ok=True)

    html_path = "figures/temp_heatmap.html"
    m.save(html_path)

    hti = Html2Image(output_path="figures")
    hti.screenshot(
        html_file=html_path,
        save_as=output_name,
        size=(1200, 800)
    )


In [ ]:
get_heat_map(df_obs,11)